# Course 6: From Beginner to an AI Research Lab
## Session 9: Experiments that convince

**Goal:** measure noise across seeds, report honest intervals, and see why a baseline must get the same tuning budget as your method.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. Five runs, one honest number
Five made-up accuracies. Work it by hand first: the mean, the sample standard deviation (divide by n − 1), the standard error, and the 95% interval using the t distribution with 4 degrees of freedom.

```
            n
mean = (1/n) Σ x_i          se = s / sqrt(n)          interval = mean ± t · se
           i=1
```

In [ ]:
import numpy as np
from scipy import stats

runs = np.array([70.1, 71.9, 71.0, 72.4, 70.6])
mean = runs.mean()
se = runs.std(ddof=1) / np.sqrt(len(runs))
half = stats.t.ppf(0.975, len(runs) - 1) * se
print(mean, se, mean - half, mean + half)

rng = np.random.default_rng(0)
boot = [rng.choice(runs, len(runs)).mean() for _ in range(10000)]
print(np.percentile(boot, [2.5, 97.5]))
assert abs(mean - 71.2) < 1e-9 and abs(se - 0.42) < 0.005 and abs(half - 1.17) < 0.01

The mean is 71.2, the standard error 0.42, and the t-interval about 70.0 to 72.4. The bootstrap interval is narrower because a bootstrap on only five values understates the uncertainty. With so few runs, report the t-interval.

### 2. The mistake of leaving out `ddof=1`

In [ ]:
print("std with ddof=1 (correct for a sample): %.3f | with the default ddof=0: %.3f" % (runs.std(ddof=1), runs.std()))

### 3. Is method B better than method A?
Two methods, five seeds each. Overlapping intervals do not settle it, and a difference in means alone does not either. If both methods were run on the same seeds, compare them run by run (a paired comparison), which removes the variation the seeds share.

In [ ]:
rng = np.random.default_rng(1)
seed_effect = rng.normal(0, 1.0, 5)                           # some seeds are lucky for every method
A = 70.0 + seed_effect + rng.normal(0, 0.3, 5); B = 70.8 + seed_effect + rng.normal(0, 0.3, 5)
ci = lambda v: stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
print(f"A: {A.mean():.2f} ± {ci(A):.2f}   B: {B.mean():.2f} ± {ci(B):.2f}   (the intervals overlap)")
d = B - A
print(f"paired difference B - A: {d.mean():.2f} ± {ci(d):.2f}   p = {stats.ttest_rel(B, A).pvalue:.4f}")
print(f"unpaired test (ignores that the seeds are shared): p = {stats.ttest_ind(B, A).pvalue:.3f}")
assert stats.ttest_rel(B, A).pvalue < stats.ttest_ind(B, A).pvalue

### 4. A fair tuning budget
Suppose your method and the baseline are in truth equally good. You try 8 settings of your method and keep the best, but run the baseline once with its defaults. How often do you 'win'?

In [ ]:
rng = np.random.default_rng(0); wins_unfair, wins_fair, gains = 0, 0, []
for _ in range(5000):
    mine = rng.normal(70, 1, 8); base = rng.normal(70, 1, 8)          # 8 settings each; all are equally good in truth
    wins_unfair += mine.max() > base[0]; wins_fair += mine.max() > base.max(); gains.append(mine.max() - base[0])
print(f"8 settings for my method, 1 for the baseline: I 'win' {wins_unfair / 5000:.0%} of the time, by {np.mean(gains):.2f} points on average")
print(f"8 settings each: I 'win' {wins_fair / 5000:.0%} of the time")
assert wins_unfair / 5000 > 0.8 and abs(wins_fair / 5000 - 0.5) < 0.03

A gain of more than a point from nothing but unequal tuning. Fix a budget, give both sides the same budget, choose on validation data, and then report on the test set with fresh seeds.

### 5. The results table and the plot
One row per method: mean, 95% interval, number of runs. Then plot every run as a dot so the spread is visible.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
results = {"baseline (tuned, 8 configs)": A, "our method (tuned, 8 configs)": B, "ablation: without component X": B - 0.5 + rng.normal(0, 0.3, 5)}
table = pd.DataFrame([{"method": k, "mean": round(v.mean(), 2), "ci95": round(ci(v), 2), "runs": len(v)} for k, v in results.items()])
print(table.to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3))
for i, (k, v) in enumerate(results.items()):
    ax.plot(v, [i] * len(v), "o", alpha=0.6); ax.plot([v.mean()], [i], "k|", markersize=18)
ax.set_yticks(range(len(results))); ax.set_yticklabels(list(results)); ax.set_xlabel("accuracy (%), 5 seeds each; bar = mean"); fig.tight_layout(); fig.savefig("runs.png")

### 6. From your own log
If you logged one row per run in `results/runs.csv` (session 3), the table is two lines.

In [ ]:
log = pd.DataFrame({"method": np.repeat(list(results), 5), "seed": list(range(5)) * 3, "accuracy": np.concatenate(list(results.values()))})
log.to_csv("runs.csv", index=False)
summary = pd.read_csv("runs.csv").groupby("method").accuracy.agg(["mean", "std", "count"])
summary["ci95"] = stats.t.ppf(0.975, summary["count"] - 1) * summary["std"] / np.sqrt(summary["count"])
print(summary.round(2))

### 7. Three sentences
Write what you can conclude, what you cannot, and what you would run next. Example: 'Over five shared seeds our method is 0.8 points above the tuned baseline (paired 95% interval given above). Removing component X costs about half of that gain. Five seeds on one dataset do not show whether this holds at larger scale.'

### TODO
1. Launch your capstone experiment over five seeds and build the table from your own log.
2. Tune the baseline with the same budget as your method.
3. Hand in the table, the dot plot and the three sentences.